# GoldenMatch GPU Endpoint (Colab)

Run this notebook on Google Colab with a GPU to provide a free remote
embedding endpoint for GoldenMatch.

**Setup:**
1. Open in Colab: Runtime > Change runtime type > T4 GPU
2. Run all cells
3. Copy the ngrok URL printed at the bottom
4. On your local machine:
```bash
export GOLDENMATCH_GPU_ENDPOINT=<the_url>
goldenmatch dedupe your_file.csv
```

In [ ]:
!pip install -q sentence-transformers flask pyngrok

In [ ]:
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models = {}

def get_model(name='all-MiniLM-L6-v2'):
    if name not in models:
        print(f'Loading {name} on {device}...')
        models[name] = SentenceTransformer(name, device=device)
    return models[name]

get_model()
print('Model ready.')

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'device': device, 'models': list(models.keys())})

@app.route('/embed', methods=['POST'])
def embed():
    data = request.json
    model = get_model(data.get('model', 'all-MiniLM-L6-v2'))
    embeddings = model.encode(data['texts'], show_progress_bar=False, normalize_embeddings=True)
    return jsonify({'embeddings': embeddings.tolist(), 'count': len(data['texts'])})

threading.Thread(target=lambda: app.run(port=5000), daemon=True).start()
print('Server running on port 5000')

In [ ]:
from pyngrok import ngrok

# Get a free ngrok token at https://ngrok.com (sign up required)
# Uncomment and set your token:
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')

url = ngrok.connect(5000)
print(f'\n{"="*50}')
print(f'GoldenMatch GPU Endpoint is live!')
print(f'URL: {url}')
print(f'\nOn your machine:')
print(f'  export GOLDENMATCH_GPU_ENDPOINT={url}')
print(f'  goldenmatch dedupe your_file.csv')
print(f'{"="*50}')

In [ ]:
# Test it
import requests, numpy as np

r = requests.get(f'{url}/health')
print('Health:', r.json())

r = requests.post(f'{url}/embed', json={
    'texts': ['John Smith', 'Jon Smith', 'Jane Doe'],
})
emb = np.array(r.json()['embeddings'])
sim = emb @ emb.T
print(f'John/Jon similarity: {sim[0][1]:.3f}')
print(f'John/Jane similarity: {sim[0][2]:.3f}')

In [ ]:
# Keep alive
import time
print('Endpoint running. Keep this notebook open.')
while True:
    time.sleep(60)